In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import train_test_split
from scipy.stats import spearmanr
import optuna

import copy

# PyG
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GCNConv, global_mean_pool

# NUPACK
from nupack import Model, pairs

np.set_printoptions(threshold=np.inf)
print("Using PyTorch and PyG.")

########################################
# 1. Data Loading
########################################

def load_branch1_data():
    """
    Loads (160,4) data from "Feature_CNN1.txt".
    """
    data_branch1 = []
    current_array = []
    with open("Feature_CNN1.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch1.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch1.append(current_array)
    X_branch1 = np.array(data_branch1)
    X_branch1 = X_branch1.reshape(len(X_branch1), 160, 4)
    return X_branch1

def load_branch2_data():
    """
    Loads (280,3) data from "Feature_CNN2.txt".
    """
    data_branch2 = []
    current_array = []
    with open("Feature_CNN2.txt", 'r') as f:
        for line in f:
            line = line.strip()
            if line == "":
                if current_array:
                    data_branch2.append(current_array)
                    current_array = []
            else:
                line = line.replace('[','').replace(']', '')
                row = [float(x.strip().replace(',', '')) for x in line.split()]
                current_array.append(row)
    if current_array:
        data_branch2.append(current_array)
    X_branch2 = np.array(data_branch2)
    X_branch2 = X_branch2.reshape(len(X_branch2), 280, 3)
    return X_branch2

def load_branch3_data(filename="Feature_MLP.txt"):
    """
    Loads 17-dimensional numeric features from file.
    Each line has 17 floats (space- or comma-delimited).
    """
    X_branch3 = []
    with open(filename, 'r') as file:
        for line in file:
            line = line.strip()
            # Expect 17 numbers per line
            values = line.replace('[', '').replace(']', '').split()
            float_vals = [float(v.strip().replace(',', '')) for v in values]
            X_branch3.append(float_vals)
    
    X_branch3 = np.array(X_branch3)  # shape: (n_samples, 17)
    X_branch3 = X_branch3.reshape(len(X_branch3), 17)
    return X_branch3

def load_reaction_rates():
    with open('HTCas9_indel_frequency_value_percentage.txt', 'r') as f:
        rates = [float(line.strip()) for line in f.readlines()]
    return np.array(rates)

########################################
# 2. Graph Data Utilities (for GNN)
########################################

def nucleotide_to_one_hot(nucleotide):
    mapping = {
        'A': [1, 0, 0, 0],
        'C': [0, 1, 0, 0],
        'G': [0, 0, 1, 0],
        'T': [0, 0, 0, 1],
        'U': [0, 0, 0, 1],
        '-': [0, 0, 0, 0]
    }
    return mapping.get(nucleotide, [0,0,0,0])

def encode_sequence(sequence):
    return [nucleotide_to_one_hot(nuc) for nuc in sequence]

def matrix_to_edge_index(prob_matrix):
    edges_source = []
    edges_target = []
    L = len(prob_matrix)
    for i in range(L):
        for j in range(i + 1, L):  # only i < j
            if prob_matrix[i][j] > 0.01:
                edges_source.append(i)
                edges_target.append(j)
    return [edges_source, edges_target]

def generate_edge_features(edge_index, seq_one_hot, prob_matrix):
    features = []
    if not edge_index:
        return []
    for i in range(len(edge_index[0])):
        src_idx = edge_index[0][i]
        tgt_idx = edge_index[1][i]
        prob_val = prob_matrix[src_idx][tgt_idx]
        
        def decode_nt(one_hot_vec):
            if one_hot_vec[0] == 1:
                return 'A'
            elif one_hot_vec[1] == 1:
                return 'C'
            elif one_hot_vec[2] == 1:
                return 'G'
            elif one_hot_vec[3] == 1:
                return 'T'
            return 'N'
        
        src_char = decode_nt(seq_one_hot[src_idx])
        tgt_char = decode_nt(seq_one_hot[tgt_idx])
        
        if (src_char == 'A' and tgt_char in ['T','U']) or (src_char in ['T','U'] and tgt_char == 'A'):
            pair_type = 2
        elif (src_char == 'G' and tgt_char in ['T','U']) or (src_char in ['T','U'] and tgt_char == 'G'):
            pair_type = 2
        elif (src_char == 'C' and tgt_char == 'G') or (src_char == 'G' and tgt_char == 'C'):
            pair_type = 3
        
        features.append([pair_type/3, prob_val])
    return features

# NUPACK models
from nupack import Model, pairs
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def load_dna_reaction_data():
    with open('HTCas9_full_guide_sequences.txt', 'r') as f:
        gRNA_sequences = [line.strip() for line in f.readlines()]
    with open('HTCas9_target_sequences_noPAM.txt', 'r') as f:
        target_sequences = [line.strip() for line in f.readlines()]
    
    gRNA_node_features = [encode_sequence(seq) for seq in gRNA_sequences]
    
    target_bh_sequences = [target_sequences[i] + DNA_reverse_complement(target_sequences[i]) for i in range(len(target_sequences))]
    target_bh_node_features = [encode_sequence(seq) for seq in target_bh_sequences]
    
    duplex_sequences = [gRNA_sequences[i] + target_sequences[i] for i in range(len(target_sequences))]
    duplex_node_features = [encode_sequence(seq) for seq in duplex_sequences]

    target_ah_node_features = [encode_sequence(DNA_reverse_complement(seq)) for seq in target_sequences]

    gRNA_edge_indices = []
    gRNA_edge_features = []
    for guide in gRNA_sequences:
        prob_matrix = pairs(strands=guide, model=my_model_RNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        oh_guide = [nucleotide_to_one_hot(n) for n in guide]
        feats = generate_edge_features(edge_index, oh_guide, prob_matrix)
        gRNA_edge_indices.append(edge_index)
        gRNA_edge_features.append(feats)

    target_bh_edge_indices = []
    target_bh_edge_features = []
    for i in range (0, len(gRNA_sequences)):
        prob_matrix = pairs(strands=[target_sequences[i], DNA_reverse_complement(target_sequences[i])], model=my_model_DNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        oh_duplex = [nucleotide_to_one_hot(n) for n in target_bh_sequences[i]]
        feats = generate_edge_features(edge_index, oh_duplex, prob_matrix)
        target_bh_edge_indices.append(edge_index)
        target_bh_edge_features.append(feats)
    
    duplex_edge_indices = []
    duplex_edge_features = []
    for i in range (0, len(gRNA_sequences)):
        prob_matrix = pairs(strands=[gRNA_sequences[i], target_sequences[i]], model=my_model_DNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        oh_duplex = [nucleotide_to_one_hot(n) for n in duplex_sequences[i]]
        feats = generate_edge_features(edge_index, oh_duplex, prob_matrix)
        duplex_edge_indices.append(edge_index)
        duplex_edge_features.append(feats)

    target_ah_edge_indices = []
    target_ah_edge_features = [] 
    for target in target_sequences:
        prob_matrix = pairs(strands=DNA_reverse_complement(target), model=my_model_DNA).to_array()
        edge_index = matrix_to_edge_index(prob_matrix)
        oh_guide = [nucleotide_to_one_hot(n) for n in DNA_reverse_complement(target)]
        feats = generate_edge_features(edge_index, oh_guide, prob_matrix)
        target_ah_edge_indices.append(edge_index)
        target_ah_edge_features.append(feats)
    
    with open('HTCas9_indel_frequency_value_percentage.txt', 'r') as f:
        reaction_rates = [float(line.strip()) for line in f.readlines()]
    
    return (gRNA_node_features, target_bh_node_features, duplex_node_features, target_ah_node_features,
            gRNA_edge_indices, target_bh_edge_indices, duplex_edge_indices, target_ah_edge_indices,
            gRNA_edge_features, target_bh_edge_features, duplex_edge_features, target_ah_edge_features,
            reaction_rates)

class GNNDataset(Dataset):
    def __init__(self, gRNA_node_features, target_bh_node_features, duplex_node_features, target_ah_node_features,
                 gRNA_edge_indices, target_bh_edge_indices, duplex_edge_indices, target_ah_edge_indices,
                 gRNA_edge_features, target_bh_edge_features, duplex_edge_features, target_ah_edge_features,
                 reaction_rates):
        super().__init__()
        self.gRNA_node_features = gRNA_node_features
        self.target_bh_node_features = target_bh_node_features
        self.duplex_node_features = duplex_node_features
        self.target_ah_node_features = target_ah_node_features
        self.gRNA_edge_indices = gRNA_edge_indices
        self.target_bh_edge_indices = target_bh_edge_indices
        self.duplex_edge_indices = duplex_edge_indices
        self.target_ah_edge_indices = target_ah_edge_indices
        self.gRNA_edge_features = gRNA_edge_features
        self.target_bh_edge_features = target_bh_edge_features
        self.duplex_edge_features = duplex_edge_features
        self.target_ah_edge_features = target_ah_edge_features
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
    def __len__(self):
        return self.num_samples
    def __getitem__(self, idx):
        c_x = torch.tensor(self.gRNA_node_features[idx], dtype=torch.float)
        c_edge_index_list = self.gRNA_edge_indices[idx]
        if not c_edge_index_list:
            c_edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            c_edge_index = torch.tensor(c_edge_index_list, dtype=torch.long)
        c_edge_attr = torch.tensor(self.gRNA_edge_features[idx], dtype=torch.float)
        if c_edge_attr.dim() == 1:
            c_edge_attr = c_edge_attr.unsqueeze(-1).repeat(1,2)
        gRNA_data = Data(x=c_x, edge_index=c_edge_index, edge_attr=c_edge_attr)

        t_bh_x = torch.tensor(self.target_bh_node_features[idx], dtype=torch.float)
        t_bh_edge_index_list = self.target_bh_edge_indices[idx]
        if not t_bh_edge_index_list:
            t_bh_edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            t_bh_edge_index = torch.tensor(t_bh_edge_index_list, dtype=torch.long)
        t_bh_edge_attr = torch.tensor(self.target_bh_edge_features[idx], dtype=torch.float)
        if t_bh_edge_attr.dim() == 1:
            t_bh_edge_attr = t_bh_edge_attr.unsqueeze(-1).repeat(1,2)
        target_bh_data = Data(x=t_bh_x, edge_index=t_bh_edge_index, edge_attr=t_bh_edge_attr)
        
        d_x = torch.tensor(self.duplex_node_features[idx], dtype=torch.float)
        d_edge_index_list = self.duplex_edge_indices[idx]
        if not d_edge_index_list:
            d_edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            d_edge_index = torch.tensor(d_edge_index_list, dtype=torch.long)
        d_edge_attr = torch.tensor(self.duplex_edge_features[idx], dtype=torch.float)
        if d_edge_attr.dim() == 1:
            d_edge_attr = d_edge_attr.unsqueeze(-1).repeat(1,2)
        duplex_data = Data(x=d_x, edge_index=d_edge_index, edge_attr=d_edge_attr)

        t_ah_x = torch.tensor(self.target_ah_node_features[idx], dtype=torch.float)
        t_ah_edge_index_list = self.target_ah_edge_indices[idx]
        if not t_ah_edge_index_list:
            t_ah_edge_index = torch.empty((2,0), dtype=torch.long)
        else:
            t_ah_edge_index = torch.tensor(t_ah_edge_index_list, dtype=torch.long)
        t_ah_edge_attr = torch.tensor(self.target_ah_edge_features[idx], dtype=torch.float)
        if t_ah_edge_attr.dim() == 1:
            t_ah_edge_attr = t_ah_edge_attr.unsqueeze(-1).repeat(1,2)
        target_ah_data = Data(x=t_ah_x, edge_index=t_ah_edge_index, edge_attr=t_ah_edge_attr)
        
        y = torch.tensor([self.reaction_rates[idx]], dtype=torch.float)
        return gRNA_data, target_bh_data, duplex_data, target_ah_data, y

def collate(batch):
    from torch_geometric.data import Batch
    gRNA_list, target_bh_list, duplex_list, target_ah_list, y_list = zip(*batch)
    batch_gRNA = Batch.from_data_list(gRNA_list)
    batch_target_bh = Batch.from_data_list(target_bh_list)
    batch_duplex = Batch.from_data_list(duplex_list)
    batch_target_ah = Batch.from_data_list(target_ah_list)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return batch_gRNA, batch_target_bh, batch_duplex, batch_target_ah, y

########################################
# 3. CNN, GNN, MLP Branches (One hidden MLP)
########################################

class CNNBranch1(nn.Module):
    """
    CNN branch for (160,4).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(4, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*80, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,4,80)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,80)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x

class CNNBranch2(nn.Module):
    """
    CNN branch for (280,3).
    """
    def __init__(self, filters, kernel_size, dense_units):
        super().__init__()
        self.conv = nn.Conv1d(3, filters, kernel_size, padding=(kernel_size-1)//2)
        self.pool = nn.MaxPool1d(2)
        self.fc = nn.Linear(filters*140, dense_units)
    def forward(self, x):
        x = x.permute(0,2,1)  # (batch,3,280)
        x = F.relu(self.conv(x))
        x = self.pool(x)      # (batch,filters,140)
        x = x.flatten(start_dim=1)
        x = F.relu(self.fc(x))
        return x

from torch_geometric.nn import NNConv, global_mean_pool, global_max_pool


class GNNBranch(nn.Module):
    def __init__(self, hidden_dim=32, edge_attr_dim=2):
        super().__init__()
        self.hidden_dim = hidden_dim

        def make_edge_mlp(edge_attr_dim, in_channels, out_channels):
            return nn.Sequential(
                nn.Linear(edge_attr_dim, hidden_dim),
                nn.ReLU(),
                nn.Linear(hidden_dim, in_channels * out_channels)
            )


        # gRNA
        self.edge_mlp_c = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_c2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_c = NNConv(4, hidden_dim, self.edge_mlp_c, aggr='mean')
        self.conv2_c = NNConv(hidden_dim, hidden_dim, self.edge_mlp_c2, aggr='mean')

        # target_bh
        self.edge_mlp_bh = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_bh2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_t_bh = NNConv(4, hidden_dim, self.edge_mlp_bh, aggr='mean')
        self.conv2_t_bh = NNConv(hidden_dim, hidden_dim, self.edge_mlp_bh2, aggr='mean')


        # duplex
        self.edge_mlp_d = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_d2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_d = NNConv(4, hidden_dim, self.edge_mlp_d, aggr='mean')
        self.conv2_d = NNConv(hidden_dim, hidden_dim, self.edge_mlp_d2, aggr='mean')

        # target_ah
        self.edge_mlp_ah = make_edge_mlp(edge_attr_dim, 4, hidden_dim)
        self.edge_mlp_ah2 = make_edge_mlp(edge_attr_dim, hidden_dim, hidden_dim)
        self.conv1_t_ah = NNConv(4, hidden_dim, self.edge_mlp_ah, aggr='mean')
        self.conv2_t_ah = NNConv(hidden_dim, hidden_dim, self.edge_mlp_ah2, aggr='mean')


        # MLP after pooling all graph embeddings
        self.mlp_merge = nn.Sequential(
            nn.Linear(hidden_dim * 4 * 2, hidden_dim),  # 4 graphs, mean+max
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

    def pool(self, x, batch):
        return torch.cat([
            global_mean_pool(x, batch),
            global_max_pool(x, batch)
        ], dim=1)


    def forward(self, gRNA_data, target_bh_data, duplex_data, target_ah_data):
        def process(graph_data, conv1, conv2):
            x, edge_index, edge_attr, batch = (
                graph_data.x, graph_data.edge_index, graph_data.edge_attr, graph_data.batch
            )
            x = F.elu(conv1(x, edge_index, edge_attr))
            x = F.elu(conv2(x, edge_index, edge_attr))
            return self.pool(x, batch)

        x_c = process(gRNA_data, self.conv1_c, self.conv2_c)
        x_t_bh = process(target_bh_data, self.conv1_t_bh, self.conv2_t_bh)
        x_d = process(duplex_data, self.conv1_d, self.conv2_d)
        x_t_ah = process(target_ah_data, self.conv1_t_ah, self.conv2_t_ah)

        merged = torch.cat([x_c, x_t_bh, x_d, x_t_ah], dim=1)
        return self.mlp_merge(merged)


class MLPBranch17(nn.Module):
    """
    A single hidden layer MLP for 17D => output dimension mlp_dim.
    No separate "output" layer; just one fc + ReLU => final embedding.
    """
    def __init__(self, input_dim=17, hidden_dim=32):
        super().__init__()
        self.fc = nn.Linear(input_dim, hidden_dim)
    def forward(self, x):
        # x shape: (batch,17)
        x = F.relu(self.fc(x))  # (batch,hidden_dim)
        return x

########################################
# 4. Final Fusion Model
########################################

class CNN_GNN_MLP_Fusion(nn.Module):
    """
    End-to-end: 
      - CNNBranch1 => feat_cnn1
      - CNNBranch2 => feat_cnn2
      - GNNBranch  => feat_gnn
      - MLPBranch17 => feat_mlp
    Concat => dropout => final FC => 1
    """
    def __init__(self,
                 filters1, kernel_size1, dense_units1,  # CNN1
                 filters2, kernel_size2, dense_units2,  # CNN2
                 gnn_hidden_dim,
                 mlp_hidden_dim,  # single hidden dimension for MLP
                 final_fc_dim,
                 dropout_rate=0.0):  # new hyperparameter for dropout
        super().__init__()
        
        self.cnn_branch1 = CNNBranch1(filters1, kernel_size1, dense_units1)
        self.cnn_branch2 = CNNBranch2(filters2, kernel_size2, dense_units2)
        self.gnn_branch = GNNBranch(hidden_dim=gnn_hidden_dim)
        self.mlp_branch = MLPBranch17(input_dim=17, hidden_dim=mlp_hidden_dim)
        
        # total dimension = (dense_units1 + dense_units2 + gnn_hidden_dim + mlp_hidden_dim)
        total_dim = dense_units1 + dense_units2 + gnn_hidden_dim + mlp_hidden_dim
        
        self.dropout_rate = dropout_rate
        self.hidden = nn.Linear(total_dim, final_fc_dim)
        self.out = nn.Linear(final_fc_dim, 1)

    
    def forward(self, gRNA_data, target_bh_data, duplex_data, target_ah_data, x1, x2, x3):
        feat_cnn1 = self.cnn_branch1(x1)                   # (batch, dense_units1)
        feat_cnn2 = self.cnn_branch2(x2)                   # (batch, dense_units2)
        feat_gnn = self.gnn_branch(gRNA_data, target_bh_data, duplex_data, target_ah_data)  # (batch, gnn_hidden_dim)
        feat_mlp = self.mlp_branch(x3)                 # (batch, mlp_hidden_dim)
        
        merged = torch.cat([feat_cnn1, feat_cnn2, feat_gnn, feat_mlp], dim=1)
        # Apply dropout on the concatenated features
        merged = F.dropout(merged, p=self.dropout_rate, training=self.training)
        x = F.relu(self.hidden(merged))   # (batch, final_fc_dim)
        out = self.out(x)                 # (batch, 1)
        return out.view(-1)


########################################
# 5. Hybrid Dataset
########################################

class HybridDataset(Dataset):
    def __init__(self, graph_dataset, X1, X2, X3, reaction_rates):
        super().__init__()
        self.graph_dataset = graph_dataset
        self.X1 = X1
        self.X2 = X2
        self.X3 = X3
        self.reaction_rates = reaction_rates
        self.num_samples = len(reaction_rates)
        
        assert len(graph_dataset) == self.num_samples
        assert len(X1) == self.num_samples
        assert len(X2) == self.num_samples
        assert len(X3) == self.num_samples
    
    def __len__(self):
        return self.num_samples
    
    def __getitem__(self, idx):
        gRNA_data, target_bh_data, duplex_data, target_ah_data, y_val = self.graph_dataset[idx]
        x1 = torch.tensor(self.X1[idx], dtype=torch.float)  
        x2 = torch.tensor(self.X2[idx], dtype=torch.float)  
        x3 = torch.tensor(self.X3[idx], dtype=torch.float)  
        return gRNA_data, target_bh_data, duplex_data, target_ah_data, x1, x2, x3, y_val

def hybrid_collate(batch):
    from torch_geometric.data import Batch
    gRNA_list, target_bh_list, duplex_list, target_ah_list, x1_list, x2_list, x3_list, y_list = zip(*batch)
    batch_gRNA = Batch.from_data_list(gRNA_list)
    batch_target_bh = Batch.from_data_list(target_bh_list)
    batch_duplex = Batch.from_data_list(duplex_list)
    batch_target_ah = Batch.from_data_list(target_ah_list)
    x1 = torch.stack(x1_list, dim=0)
    x2 = torch.stack(x2_list, dim=0)
    x3 = torch.stack(x3_list, dim=0)
    y = torch.tensor(y_list, dtype=torch.float).view(-1)
    return batch_gRNA, batch_target_bh, batch_duplex, batch_target_ah, x1, x2, x3, y

########################################
# 6. Objective Function (no pretraining)
########################################

import copy

# Global tracking across trials
best_trial_val_loss = float("inf")
best_model_global = None

def objective(trial):
    # CNN1
    filters1 = trial.suggest_int("filters1", 16, 128, step=16)
    kernel_size1 = trial.suggest_categorical("kernel_size1", [3,5,7])
    dense_units1 = trial.suggest_int("dense_units1", 16, 256, step=16)
    
    # CNN2
    filters2 = trial.suggest_int("filters2", 16, 128, step=16)
    kernel_size2 = trial.suggest_categorical("kernel_size2", [3,5,7])
    dense_units2 = trial.suggest_int("dense_units2", 16, 256, step=16)
    
    # GNN
    gnn_hidden_dim = trial.suggest_int("gnn_hidden_dim", 16, 64, step=16)
    
    # MLP: single hidden layer => "mlp_hidden_dim"
    mlp_hidden_dim = trial.suggest_int("mlp_hidden_dim", 16, 256, step=16)
    
    # final aggregator
    final_fc_dim = trial.suggest_int("final_fc_dim", 16, 256, step=16)

    # Dropout hyperparameter (0.2 to 0.7)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    
    # Learning rate and batch size
    lr = trial.suggest_loguniform("lr", 1e-4, 1e-2)
    batch_size = trial.suggest_categorical("batch_size", [8,16,32])
    epochs = 100
    patience = 10
    
    loss_fn = nn.MSELoss()
    
    # load data
    X1 = load_branch1_data()
    X2 = load_branch2_data()
    X3 = load_branch3_data()
    rates = load_reaction_rates()
    
    (gRNA_node_feats, target_bh_node_feats, duplex_node_feats, target_ah_node_feats,
     gRNA_edge_indices, target_bh_edge_indices, duplex_edge_indices, target_ah_edge_indices,
     gRNA_edge_features, target_bh_edge_features, duplex_edge_features, target_ah_edge_feature,
     _) = load_dna_reaction_data()
    graph_dataset = GNNDataset(
        gRNA_node_feats, target_bh_node_feats, duplex_node_feats, target_ah_node_feats,
        gRNA_edge_indices, target_bh_edge_indices, duplex_edge_indices, target_ah_edge_indices,
        gRNA_edge_features, target_bh_edge_features, duplex_edge_features, target_ah_edge_feature,
        rates
    )
    hybrid_dataset = HybridDataset(graph_dataset, X1, X2, X3, rates)
    
    # Splitting the dataset
    np.random.seed(42)
    full_indices = np.arange(len(rates))
    selected_indices = np.random.choice(len(full_indices), size=10699, replace=False)
    unseen_indices = np.setdiff1d(full_indices, selected_indices)
    train_idx, val_idx = train_test_split(selected_indices, test_size=0.2, random_state=42)
    
    train_set = Subset(hybrid_dataset, train_idx)
    val_set = Subset(hybrid_dataset, val_idx)
    
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True, collate_fn=hybrid_collate)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False, collate_fn=hybrid_collate)
    
    # Build the model
    model = CNN_GNN_MLP_Fusion(
        filters1, kernel_size1, dense_units1,
        filters2, kernel_size2, dense_units2,
        gnn_hidden_dim,
        mlp_hidden_dim,
        final_fc_dim,
        dropout_rate=dropout_rate
    )
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    
    best_val_loss = float('inf')
    counter = 0
    best_model_state = None  # To store the best model's state
    
    for epoch in range(epochs):
        # Training phase
        model.train()
        total_loss = 0.0
        for gRNA_b, target_bh_b, duplex_b, target_ah_b, x1_b, x2_b, x3_b, y_b in train_loader:
            pred = model(gRNA_b, target_bh_b, duplex_b, target_ah_b, x1_b, x2_b, x3_b)
            loss = loss_fn(pred, y_b.view(-1))
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        avg_train_loss = total_loss / len(train_loader)
        
        # Validation phase
        model.eval()
        total_val = 0.0
        with torch.no_grad():
            for gRNA_b, target_bh_b, duplex_b, target_ah_b, x1_b, x2_b, x3_b, y_b in val_loader:
                valpred = model(gRNA_b, target_bh_b, duplex_b, target_ah_b, x1_b, x2_b, x3_b)
                vloss = loss_fn(valpred, y_b.view(-1))
                total_val += vloss.item()
        avg_val_loss = total_val / len(val_loader)
        
        trial.report(avg_val_loss, epoch)
        
        # Check for improvement
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_model_state = copy.deepcopy(model)
            counter = 0
        else:
            counter += 1
        
        # Early stopping if no improvement for 'patience' epochs
        if counter >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break
        
        print(f"Epoch {epoch+1}/{epochs} | Train: {avg_train_loss:.4f} | Val: {avg_val_loss:.4f}")
    
    # Load best model from this trial
    if best_model_state is not None:
        model = best_model_state
    
    # Save only if this trial is the best across ALL trials
    global best_trial_val_loss
    if best_val_loss < best_trial_val_loss:
        best_trial_val_loss = best_val_loss
        torch.save(model, "best_FULL.pt")
        print(f"New best model saved with val_loss = {best_val_loss:.6f}")

    unseen_set = Subset(hybrid_dataset, unseen_indices)
    model.eval()
    trial_loader = DataLoader(unseen_set, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    preds_trial = []
    labels_trial = []
    with torch.no_grad():
        for g_b, t_bh_b, d_b, t_ah_b, x1_b, x2_b, x3_b, y_b in trial_loader:
            p = model(g_b, t_bh_b, d_b, t_ah_b, x1_b, x2_b, x3_b)
            preds_trial.append(p.item())
            labels_trial.append(y_b.item())
    sp_corr, _ = spearmanr(labels_trial, preds_trial)
    print(f"Spearman for best model = {sp_corr}")
    
    return best_val_loss

########################################
# 7. Main Pipeline
########################################

def main_pipeline():
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=100)
    
    print("Best hyperparameters:", study.best_trial.params)
    best_trial_number = study.best_trial.number
    best_params = study.best_trial.params
    
    # Rebuild for final 100-trial evaluation by loading the entire saved model
    model = torch.load(f"best_FULL.pt", weights_only=False)
    model.eval()
    
    # final hold-out evaluation
    X1 = load_branch1_data()
    X2 = load_branch2_data()
    X3 = load_branch3_data()
    y = load_reaction_rates()
    
    (gRNA_node_feats, target_bh_node_feats, duplex_node_feats, target_ah_node_feats,
     gRNA_edge_indices, target_bh_edge_indices, duplex_edge_indices, target_ah_edge_indices,
     gRNA_edge_features, target_bh_edge_features, duplex_edge_features, target_ah_edge_features,
     _) = load_dna_reaction_data()
    graph_dataset = GNNDataset(gRNA_node_feats, target_bh_node_feats, duplex_node_feats, target_ah_node_feats,
                                                   gRNA_edge_indices, target_bh_edge_indices, duplex_edge_indices, target_ah_edge_indices,
                                                   gRNA_edge_features, target_bh_edge_features, duplex_edge_features, target_ah_edge_features,
                                                   y)
    hybrid_dataset = HybridDataset(graph_dataset, X1, X2, X3, y)
    
    np.random.seed(42)
    full_indices = np.arange(len(y))
    selected_indices = np.random.choice(len(full_indices), size=10699, replace=False)
    unseen_indices = np.setdiff1d(full_indices, selected_indices)
    unseen_set = Subset(hybrid_dataset, unseen_indices)
    
    trial_loader = DataLoader(unseen_set, batch_size=1, shuffle=False, collate_fn=hybrid_collate)
    preds_trial = []
    labels_trial = []
    with torch.no_grad():
        for g_b, t_bh_b, d_b, t_ah_b, x1_b, x2_b, x3_b, y_b in trial_loader:
            p = model(g_b, t_bh_b, d_b, t_ah_b, x1_b, x2_b, x3_b)
            preds_trial.append(p.item())
            labels_trial.append(y_b.item())
    sp_corr, _ = spearmanr(labels_trial, preds_trial)
    print(f"Spearman for best model = {sp_corr}")

if __name__ == "__main__":
    main_pipeline()